# HalfCourt - Model Training Notebook

**Before running:** Change the runtime to GPU.
- Runtime -> Change runtime type -> T4 GPU -> Save

**Steps:**
1. Run Cell 1 - install dependencies
2. Run Cell 2 - upload dataset (parquet file)
3. Run Cell 3 - set your parquet file path
4. Run Cell 4 - paste model code (self-contained, no imports needed)
5. Run Cell 5 - train
6. Run Cell 6 - download the saved model

In [ ]:
# Install dependencies
!pip install -q shap scikit-learn pandas numpy joblib pyarrow

In [ ]:
# Upload the dataset
from google.colab import files
uploaded = files.upload()
print('Uploaded:', list(uploaded.keys()))

In [ ]:
# Set paths
PARQUET_PATH = 'halfcourt_training.parquet'
MODEL_SAVE_PATH = 'logistic_regression.joblib'

In [ ]:
# Model code
import os
import joblib
import numpy as np
import pandas as pd
import shap
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, brier_score_loss
from sklearn.preprocessing import StandardScaler

FEATURE_COLS = [
    "ht_score_diff", "q1_diff", "q2_diff", "momentum_shift",
    "home_fg_pct", "home_3p_pct", "home_ft_pct", "home_efg_pct", "home_ts_pct",
    "away_fg_pct", "away_3p_pct", "away_ft_pct", "away_efg_pct", "away_ts_pct",
    "home_to", "away_to", "home_ast_to_ratio", "away_ast_to_ratio",
    "home_oreb_pct", "away_oreb_pct", "reb_diff",
    "home_pace_est", "away_pace_est",
    "home_bench_pts", "away_bench_pts", "home_bench_ratio", "away_bench_ratio",
    "home_starters_3foul", "away_starters_3foul", "foul_diff",
    "home_star1_pts", "home_star1_ast", "home_star_foul_flag",
    "away_star1_pts", "away_star1_ast", "away_star_foul_flag",
    "is_home", "home_rest_days", "away_rest_days",
    "home_back2back", "away_back2back", "is_playoffs",
]
LABEL_COL  = "home_win"
MARGIN_COL = "final_margin"

MLP_CFG = {
    "hidden_layers": [128, 64, 32],
    "dropout":       0.3,
    "epochs":        100,
    "learning_rate": 0.001,
    "batch_size":    64,
    "margin_lambda": 0.1,
}
TEST_SIZE    = 0.2
RANDOM_STATE = 42


class HalfCourtDNN(nn.Module):
    def __init__(self, input_dim, hidden_layers, dropout):
        super().__init__()
        self.shared = self._build_shared(input_dim, hidden_layers, dropout)
        final_dim = hidden_layers[-1]
        self.win_head    = nn.Linear(final_dim, 1)
        self.margin_head = nn.Linear(final_dim, 1)

    def _build_shared(self, input_dim, hidden_layers, dropout):
        layers, in_dim = [], input_dim
        for out_dim in hidden_layers:
            layers += [nn.Linear(in_dim, out_dim), nn.BatchNorm1d(out_dim), nn.ReLU(), nn.Dropout(dropout)]
            in_dim = out_dim
        return nn.Sequential(*layers)

    def forward(self, x):
        shared = self.shared(x)
        return torch.sigmoid(self.win_head(shared)).squeeze(1), self.margin_head(shared).squeeze(1)


class SklearnDNNWrapper:
    def __init__(self, cfg, input_dim):
        self.epochs        = cfg["epochs"]
        self.lr            = cfg["learning_rate"]
        self.batch_size    = cfg["batch_size"]
        self.margin_lambda = cfg["margin_lambda"]
        self.device        = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.scaler        = StandardScaler()
        self.model         = HalfCourtDNN(input_dim, cfg["hidden_layers"], cfg["dropout"]).to(self.device)
        print(f"Training on: {self.device}")

    def fit(self, X, y, margins=None):
        X_scaled = self.scaler.fit_transform(X).astype(np.float32)
        if margins is None:
            margins = np.zeros(len(y), dtype=np.float32)
        dataset = TensorDataset(
            torch.tensor(X_scaled),
            torch.tensor(y, dtype=torch.float32),
            torch.tensor(margins.astype(np.float32)),
        )
        loader    = DataLoader(dataset, batch_size=self.batch_size, shuffle=True)
        optimizer = torch.optim.Adam(self.model.parameters(), lr=self.lr)
        bce, mse  = nn.BCELoss(), nn.MSELoss()
        self.model.train()
        for epoch in range(self.epochs):
            total = 0.0
            for Xb, yb, mb in loader:
                Xb, yb, mb = Xb.to(self.device), yb.to(self.device), mb.to(self.device)
                optimizer.zero_grad()
                wp, mp = self.model(Xb)
                loss = bce(wp, yb) + self.margin_lambda * mse(mp, mb)
                loss.backward()
                optimizer.step()
                total += loss.item()
            if (epoch + 1) % 20 == 0:
                print(f"  Epoch {epoch+1}/{self.epochs} - loss: {total/len(loader):.4f}")
        self.model.eval()

    def predict_proba(self, X):
        X_scaled = self.scaler.transform(X).astype(np.float32)
        self.model.eval()
        with torch.no_grad():
            wp, _ = self.model(torch.tensor(X_scaled).to(self.device))
        probs = wp.cpu().numpy()
        return np.column_stack([1 - probs, probs])

    def predict_margin(self, X):
        X_scaled = self.scaler.transform(X).astype(np.float32)
        self.model.eval()
        with torch.no_grad():
            _, mp = self.model(torch.tensor(X_scaled).to(self.device))
        return mp.cpu().numpy()


def temporal_split(df, test_size):
    split_idx = int(len(df) * (1 - test_size))
    return df.iloc[:split_idx], df.iloc[split_idx:]


def compute_shap_importance(model, X_test):
    background = X_test[:100] if len(X_test) >= 100 else X_test
    def win_prob_fn(x):
        return model.predict_proba(x)[:, 1]
    explainer   = shap.KernelExplainer(win_prob_fn, background)
    shap_values = explainer.shap_values(X_test, nsamples=100)
    mean_abs    = np.abs(shap_values).mean(axis=0)
    return dict(zip(FEATURE_COLS, mean_abs))


def print_evaluation(model, X_train, X_test, y_train, y_test):
    proba = model.predict_proba(X_test)[:, 1]
    preds = (proba >= 0.5).astype(int)
    print("\n=== HalfCourt Model Evaluation ===")
    print(f"Model type     : mlp")
    print(f"Train samples  : {len(X_train)}")
    print(f"Test samples   : {len(X_test)}")
    print()
    print(f"Accuracy       : {accuracy_score(y_test, preds):.3f}")
    print(f"AUC-ROC        : {roc_auc_score(y_test, proba):.3f}")
    print(f"F1 Score       : {f1_score(y_test, preds):.3f}")
    print(f"Brier Score    : {brier_score_loss(y_test, proba):.3f}   (lower is better; 0.25 = no skill)")
    print()
    shap_imp = compute_shap_importance(model, X_test)
    sorted_feats = sorted(shap_imp.items(), key=lambda x: x[1], reverse=True)[:10]
    print("Top 10 Features by SHAP importance:")
    for rank, (feat, val) in enumerate(sorted_feats, 1):
        print(f"  {rank:2d}. {feat:<30} {val:.3f}")
    print()

In [ ]:
# Train
df = pd.read_parquet(PARQUET_PATH)
df = df.sort_values("game_date").reset_index(drop=True)
print(f"Dataset loaded: {len(df)} games, {len(df.columns)} columns")

train_df, test_df = temporal_split(df, TEST_SIZE)

X_train  = train_df[FEATURE_COLS].values.astype(np.float32)
y_train  = train_df[LABEL_COL].values.astype(np.float32)
margins  = train_df[MARGIN_COL].values.astype(np.float32)
X_test   = test_df[FEATURE_COLS].values.astype(np.float32)
y_test   = test_df[LABEL_COL].values.astype(np.float32)

model = SklearnDNNWrapper(MLP_CFG, input_dim=X_train.shape[1])
model.fit(X_train, y_train, margins=margins)

print_evaluation(model, X_train, X_test, y_train, y_test)

joblib.dump(model, MODEL_SAVE_PATH)
print(f"\nModel saved at: {MODEL_SAVE_PATH}")

In [ ]:
# Download the trained model to your computer
from google.colab import files
files.download(MODEL_SAVE_PATH)